In [66]:
import numpy as np
import pandas as pd
import json
import os
from constants import *
from run import *
from models import *
from sim import *

import torch
import gpytorch

from scipy.stats import multivariate_normal

In [61]:
ingredient_columns = [
    'd_glucose', 'sodium_citrate', 'sodium_octanoate', 'sodium_acetate',
    'sodium_benzoate', 'sodium_chloride', 'potassium_chloride',
    'mme_trace_minerals', 'urea', 'ammonium_chloride'
]
growth_column = 'y'

# Load your data globally
df = pd.read_csv('/Users/juar705/dashboard/bacterAI_dash/P_putida_AG5577_baseline/Round1/mapped_data_2025-04-15_biotek_final_od_data.csv')
df = df.dropna(subset=ingredient_columns + [growth_column])

# Define X_train and y_train globally
X_train = df[ingredient_columns].to_numpy()
y_train = df[growth_column].to_numpy()

In [ ]:
def sample_GP(model, likelihood, X, n_samples=1):
                                                                                                                                                                                                                                         
    # Convert data to tensor
    train_x = torch.tensor(X_train, dtype=torch.float)
    train_y = torch.tensor(y_train, dtype=torch.float)
    
    test_x = torch.tensor(X, dtype=torch.float32)
    pd.DataFrame(test_x.numpy(), columns=ingredient_columns).to_csv('test_x.csv', index=False)
    # Initialize likelihood and model method
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model = gpr.ExactGPModel(train_x, train_y, likelihood)
    
    # Load the state dictionaries from previously saved files
    model.load_state_dict(torch.load('/Users/juar705/dashboard/bacterAI_dash/Round1/gpr_model/gpr_model.pth'))
    likelihood.load_state_dict(torch.load('/Users/juar705/dashboard/bacterAI_dash/Round1/gpr_model/gpr_likelihood.pth'))
    
    # Set model and likelihood to evaluation mode
    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(test_x))
     
    # Get the mean and covariance
    mean = observed_pred.mean.numpy()
    cov = observed_pred.covariance_matrix.numpy()
    
    # Sample from the multivariate normal distribution
    samples = np.atleast_1d(multivariate_normal.rvs(mean, cov, size=n_samples))
    variances = np.diag(cov)
    
    return samples, variances

In [62]:
class GPRModel(Model):
    def __init__(self, model_path):
        # self.activate_R()
        self.model = []
        self.likelihood = []
        self.model_path = model_path
        self.is_trained = False
        super().__init__(self, ModelType.GPR)
        
    @classmethod
    def load_trained_models(cls, models_path):
        obj = cls(models_path)

        for filename in os.listdir(models_path):
            if "model" in filename:
                model = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE))
                obj.model.append(model)
            if "likelihood" in filename:
                likelihood = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE))
                obj.likelihood.append(likelihood)

        obj.is_trained = True
        return obj
    
    def check_path(self):
        if not os.path.exists(self.model_path):
            os.makedirs(self.model_path)

    def train(self, X_train, y_train, **kwargs):
        # X_trainR = robjects.r.matrix(
        #     X_train, nrow=X_train.shape[0], ncol=X_train.shape[1]
        # )
        # y_trainR = robjects.r.matrix(y_train, nrow=y_train.shape[0], ncol=1)
        # self.model = self.gpr_lib.train_new_GP(X_trainR, y_trainR)
        self.check_path()
        self.model, self.likelihood = gpr.train_new_GP(X_train, y_train, self.model_path, **kwargs)
        self.is_trained = True

    def evaluate(self, X, clip=True, n=1):
        # X_evalR = robjects.r.matrix(X, nrow=X.shape[0], ncol=X.shape[1])
        if not self.is_trained:
            raise Exception("GPR model needs to be trained before evaluating.")
        
        #removed gpr to obtain straight from the notebook rather than the file
        samples, variances  = sample_GP(self.model, self.likelihood, X, n)
        # Do we want to clip samples?
        if clip:
            samples = np.clip(samples, 0, 1)
        return samples, variances

In [64]:
def train_and_predict_gpr(
    config_path,
    csv_file_path,
    ingredient_columns,
    growth_column,
    output_csv_path
):

    # Load config
    with open(config_path, 'r') as file:
        config = json.load(file)

    # Load data
    df = pd.read_csv(csv_file_path)
    df = df.dropna(subset=ingredient_columns + [growth_column])
    X_train = df[ingredient_columns].to_numpy()
    y_train = df[growth_column].to_numpy()

    MODEL_TYPE = ModelType(config["model_type"])
    EXPT_FOLDER = config.get("experiment_path", ".")
    NEW_ROUND_N = config.get("new_round_n", 1)
    new_round_folder = os.path.join(EXPT_FOLDER, f"Round{NEW_ROUND_N}")
    models_folder = os.path.join(new_round_folder, f"gpr_model")

    # Train model
    model = GPRModel(models_folder)
    model.train(X_train, y_train)

    # Predict
    test_x = torch.tensor(X_train, dtype=torch.float32)
    samples, variances = model.evaluate(test_x)
    

    # Save to CSV
    output_data = pd.DataFrame({
        'y_pred': samples,
        'y_true': y_train,
        'y_var': variances,
    })
    output_data.to_csv(output_csv_path, index=False)
    print(f"Data saved to {output_csv_path}")

    # Save X_train with y_pred
    xtrain_df = pd.DataFrame(X_train, columns=ingredient_columns)
    xtrain_df['y_pred'] = samples
    xtrain_df.to_csv('/Users/juar705/dashboard/bacterAI_dash/P_putida_AG5577_baseline/Round1/x_train_with_ypredGPR.csv', index=False)
    print("X_train with y_pred saved to x_train_with_ypredGPR.csv")

   


In [65]:
train_and_predict_gpr(config_path='config.json', csv_file_path='/Users/juar705/dashboard/bacterAI_dash/P_putida_AG5577_baseline/Round1/mapped_data_2025-04-15_biotek_final_od_data.csv', ingredient_columns=ingredient_columns, growth_column=growth_column, output_csv_path='/Users/juar705/dashboard/bacterAI_dash/P_putida_AG5577_baseline/Round1/predictions_GPR.csv')

Iter 1/100 | Train loss: 1.1144
Iter 2/100 | Train loss: 1.0632
Iter 3/100 | Train loss: 1.0170
Iter 4/100 | Train loss: 0.9769
Iter 5/100 | Train loss: 0.9418
Iter 6/100 | Train loss: 0.9077
Iter 7/100 | Train loss: 0.8705
Iter 8/100 | Train loss: 0.8291
Iter 9/100 | Train loss: 0.7849
Iter 10/100 | Train loss: 0.7403
Iter 11/100 | Train loss: 0.6972
Iter 12/100 | Train loss: 0.6565
Iter 13/100 | Train loss: 0.6167
Iter 14/100 | Train loss: 0.5752
Iter 15/100 | Train loss: 0.5307
Iter 16/100 | Train loss: 0.4844
Iter 17/100 | Train loss: 0.4387
Iter 18/100 | Train loss: 0.3956
Iter 19/100 | Train loss: 0.3540
Iter 20/100 | Train loss: 0.3108
Iter 21/100 | Train loss: 0.2650
Iter 22/100 | Train loss: 0.2193
Iter 23/100 | Train loss: 0.1765
Iter 24/100 | Train loss: 0.1355
Iter 25/100 | Train loss: 0.0927
Iter 26/100 | Train loss: 0.0485
Iter 27/100 | Train loss: 0.0072
Iter 28/100 | Train loss: -0.0312
Iter 29/100 | Train loss: -0.0716
Iter 30/100 | Train loss: -0.1117
Iter 31/100 | Tr

/var/folders/rt/sp_zyvvd0j181vdwx_5rd7k00000gn/T/ipykernel_70048/2177655585.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_x = torch.tensor(X, dtype=torch.float32)
/Users/juar705/miniconda3/envs/bacterai/lib/python3.10/site-packages/gpytorch/models/exact_gp.py:296: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(
